# Medi-Script-AI: TrOCR Fine-Tuning Notebook (Kaggle)

This notebook contains the code to fine-tune the TrOCR model using free Kaggle T4 GPUs.
It downloads the dataset via the Roboflow API, parses the annotations into a Pandas DataFrame, and fine-tunes the `microsoft/trocr-base-handwritten` checkpoint.

In [ ]:
!pip install -q transformers evaluate datasets roboflow rapidfuzz accelerate python-Levenshtein jiwer sentencepiece

### Step 1: Download Dataset from Roboflow
**IMPORTANT:**
1. Go to https://universe.roboflow.com/
2. Search for "Doctors prescriptions handwriting" and click on the dataset you want.
3. Click "Download Dataset".
4. Select "COCO" format and choose "show download code".
5. Copy the Python code snippet provided by Roboflow and paste it into the cell below (replacing the placeholder code). Make sure your API key is in the snippet!

In [ ]:
import os

# ---> PASTE YOUR ROBOFLOW DOWNLOAD SNIPPET HERE <---
# Example:
# rf = Roboflow(api_key="YOUR_API_KEY")
# project = rf.workspace("WORKSPACE_NAME").project("PROJECT_NAME")
# version = project.version(1)
# dataset = version.download("coco")

### Step 2: Parse COCO Annotations into a Pandas DataFrame
TrOCR needs a dataframe with two columns: `file_name` (absolute path to image) and `text` (ground truth transcription).

In [ ]:
import json
import pandas as pd


def create_df_from_coco(coco_json_path, image_dir):
    with open(coco_json_path, "r") as f:
        data = json.load(f)

    # Map category ID to text (if the dataset uses class names as text transcriptions)
    categories = {cat["id"]: cat["name"] for cat in data["categories"]}

    records = []
    for ann in data["annotations"]:
        img_id = ann["image_id"]
        img_info = next(item for item in data["images"] if item["id"] == img_id)
        file_name = os.path.join(image_dir, img_info["file_name"])
        text = categories[ann["category_id"]]
        records.append({"file_name": file_name, "text": text})

    return pd.DataFrame(records)


# Assuming standard Roboflow export structure (variable 'dataset' comes from the Roboflow snippet)
train_df = create_df_from_coco(
    f"{dataset.location}/train/_annotations.coco.json", f"{dataset.location}/train"
)
valid_df = create_df_from_coco(
    f"{dataset.location}/valid/_annotations.coco.json", f"{dataset.location}/valid"
)

print(
    f"Loaded {len(train_df)} training samples and {len(valid_df)} validation samples."
)
train_df.head()

### Step 3: Define PyTorch Dataset for TrOCR

In [ ]:
import torch
from torch.utils.data import Dataset
from PIL import Image


class HandwritingDataset(Dataset):
    def __init__(self, df, processor, max_target_length=128):
        self.df = df
        self.processor = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        file_name = self.df.iloc[idx]["file_name"]
        text = self.df.iloc[idx]["text"]

        image = Image.open(file_name).convert("RGB")
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()

        labels = self.processor.tokenizer(
            text,
            padding="max_length",
            max_length=self.max_target_length,
            truncation=True,
        ).input_ids

        labels = [
            l if l != self.processor.tokenizer.pad_token_id else -100 for l in labels
        ]
        return {"pixel_values": pixel_values, "labels": torch.tensor(labels)}

### Step 4: Fine-Tune TrOCR

In [ ]:
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    default_data_collator,
)
import evaluate

processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.max_length = 64
model.config.early_stopping = True
model.config.no_repeat_ngram_size = 3
model.config.length_penalty = 2.0
model.config.num_beams = 4

train_dataset = HandwritingDataset(train_df, processor)
eval_dataset = HandwritingDataset(valid_df, processor)

cer_metric = evaluate.load("cer")


def compute_metrics(pred):
    labels_ids = pred.label_ids
    pred_ids = pred.predictions
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    labels_ids[labels_ids == -100] = processor.tokenizer.pad_token_id
    label_str = processor.batch_decode(labels_ids, skip_special_tokens=True)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {"cer": cer}


training_args = Seq2SeqTrainingArguments(
    output_dir="./trocr-medical-finetuned",
    predict_with_generate=True,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=16,  # Matching the paper's hyperparams
    per_device_eval_batch_size=16,
    fp16=True,
    learning_rate=2e-5,  # Matching the paper's hyperparams
    num_train_epochs=8,  # Matching the paper's hyperparams
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    report_to="none",
)

trainer = Seq2SeqTrainer(
    model=model,
    tokenizer=processor.tokenizer,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=default_data_collator,
)

trainer.train()
trainer.save_model("./trocr-medical-best")
processor.save_pretrained("./trocr-medical-best")